# Deep Learning Anomaly Detection Experiments

This notebook contains experiments for comparing deep learning models (LSTM, CNN-LSTM, Autoencoder) for anomaly detection in automotive sensor data.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Add paths for imports
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..', '..', 'ml'))

from models.lstm import LSTMModel
from models.cnn_lstm import CNNLSTMModel
from models.autoencoder import AutoencoderModel
from utils.data_loader import SequenceDataLoader, CarOBDMLDataLoader
from utils.evaluation import evaluate_deep_learning_model, optimize_threshold

print("Imports successful!")


## 1. Load Data and Create Sequences


In [ ]:
# Load data
data_path = "data/carOBD/obdiidata"
ml_loader = CarOBDMLDataLoader(data_path)
seq_loader = SequenceDataLoader(sequence_length=30)

# Load and create sequences
train_dataset, val_dataset, test_dataset = seq_loader.load_data_from_ml_loader(
    ml_loader,
    data_path=data_path,
    validation_split=0.2,
    test_split=0.1,
    fault_percentage=0.2  # For evaluation
)

print(f"Train sequences: {len(train_dataset)}")
print(f"Validation sequences: {len(val_dataset)}")
print(f"Test sequences: {len(test_dataset)}")
print(f"Sequence shape: {train_dataset[0][0].shape}")


## 2. Load Trained Models


In [ ]:
# Load trained models (if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

models = {}
model_paths = {
    'lstm': 'anomaly-detection/models/deep_learning/lstm',
    'cnn_lstm': 'anomaly-detection/models/deep_learning/cnn_lstm',
    'autoencoder': 'anomaly-detection/models/deep_learning/autoencoder'
}

for name, path in model_paths.items():
    if os.path.exists(f"{path}.pth"):
        if name == 'lstm':
            models[name] = LSTMModel.load_model(path, device=device)
        elif name == 'cnn_lstm':
            models[name] = CNNLSTMModel.load_model(path, device=device)
        elif name == 'autoencoder':
            models[name] = AutoencoderModel.load_model(path, device=device)
        print(f"Loaded {name} model")
    else:
        print(f"Model {name} not found at {path}.pth")

print(f"\nAvailable models: {list(models.keys())}")


## 3. Evaluate Models


In [ ]:
# Evaluate each model on test set
results = {}

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    test_sequences = test_dataset.sequences.numpy()
    test_labels = test_dataset.labels.numpy()
    
    result = evaluate_deep_learning_model(
        model,
        test_sequences,
        test_labels,
        threshold=None,  # Auto-optimize
        threshold_method='percentile',
        threshold_percentile=5,
        device=device
    )
    
    results[name] = result
    print(f"F1-Score: {result['metrics']['f1_score']:.4f}")
    print(f"Precision: {result['metrics']['precision']:.4f}")
    print(f"Recall: {result['metrics']['recall']:.4f}")


## 4. Compare Model Performance


In [ ]:
# Create comparison dataframe
comparison_data = []
for name, result in results.items():
    metrics = result['metrics']
    comparison_data.append({
        'Model': name.upper(),
        'F1-Score': metrics['f1_score'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'Accuracy': metrics['accuracy'],
        'ROC-AUC': metrics.get('roc_auc', 0.0)
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))


## 5. Visualize Anomaly Scores


In [ ]:
# Plot anomaly score distributions
fig, axes = plt.subplots(len(results), 1, figsize=(12, 4 * len(results)))
if len(results) == 1:
    axes = [axes]

for idx, (name, result) in enumerate(results.items()):
    scores = result['scores']
    labels = result['labels']
    threshold = result['threshold']
    
    ax = axes[idx]
    ax.hist(scores[labels == 0], bins=50, alpha=0.7, label='Normal', color='green')
    ax.hist(scores[labels == 1], bins=50, alpha=0.7, label='Anomaly', color='red')
    ax.axvline(threshold, color='blue', linestyle='--', label=f'Threshold: {threshold:.4f}')
    ax.set_xlabel('Anomaly Score')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name.upper()} - Anomaly Score Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. ROC Curves Comparison


In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 8))

for name, result in results.items():
    scores = result['scores']
    labels = result['labels']
    
    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f'{name.upper()} (AUC = {roc_auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()
